# Run ResNet18 model on multiclass(healthy, unhealthy, empty) dataset. 
Do one run with unbalanced classes and another randomly sampling for evenly distributed data.

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# --- 1. SETUP PATHS & TRANSFORMS ---
data_dir = "../data/multiclass_dataset"
batch_size = 16 # Small batch size is better for small datasets

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 2. THE 3-CLASS DATASET ---
# This automatically maps: Empty=0, Healthy=1, Unhealthy=2 (alphabetical)
full_dataset = datasets.ImageFolder(data_dir, transform=transform)
print(full_dataset.class_to_idx)
# Filter out uncertain
indices = [i for i, (path, label) in enumerate(full_dataset.imgs) 
           if full_dataset.classes[label] != 'uncertain']

dataset = torch.utils.data.Subset(full_dataset, indices)

# Split into Train (80%) and Val (20%)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_data, val_data = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

print(f"Classes found: {full_dataset.classes}")
print(f"Training on: {len(train_data)} images | Validating on: {len(val_data)} images")

model = models.resnet18(weights='IMAGENET1K_V1')

# Freeze the early layers
for param in model.parameters():
    param.requires_grad = False

# Swap the final layer for 3 classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 3) 

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

weights = torch.tensor([1.0, 1.0, 10.0]).to(device) 
criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

print(f"Model ready on device: {device}")

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
    
    epoch_loss = running_loss / len(train_data)
    
    # Simple Validation
    model.eval()
    correct = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)
    
    accuracy = correct.float() / len(val_data)
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss:.4f} | Val Acc: {accuracy:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Get model outputs
        outputs = model(inputs)
        
        # Get the index of the highest score (the prediction)
        _, preds = torch.max(outputs, 1)
        
        # Move back to CPU and convert to numpy for the matrix
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(f"Generated predictions for {len(all_preds)} images.")


In [ ]:
# Your output confirmed: {0: 0, 1: 1, 3: 2}
target_classes = ['empty', 'healthy', 'unhealthy']
original_mapping = full_dataset.class_to_idx
remap = {original_mapping[c]: i for i, c in enumerate(target_classes)}

# 2. INITIALIZE 3x3 MATRIX
plt.figure(figsize=(10, 8))
matrix = np.zeros((3, 3), dtype=int)

# 3. FILL MATRIX USING REMAPPED INDICES
for a, p in zip(all_labels, all_preds):
    # This is the magic line: it turns 3 into 2, 1 into 1, and 0 into 0
    if a in remap and p in remap:
        matrix[remap[a], remap[p]] += 1

# 4. PLOT
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_classes, yticklabels=target_classes)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: Sapling Multiclass (3-Class)')
plt.show()

# 5. RECALL CHECK
for i, name in enumerate(target_classes):
    denom = matrix[i].sum()
    recall = matrix[i, i] / denom if denom > 0 else 0
    print(f"{name.capitalize()} Recall: {recall:.2%}")